# Part 2: Vector store + LLM action plans

Run **`Index.ipynb`** first so `RAG_docs/vector_store/` exists (FAISS index + `rag_parents.json`).

**This notebook** loads the **saved vector store only** (no re-reading `RAG_docs/knowledge/`) and writes JSON under `RAG_docs/action_plans/` (including optional rerank comparison reports).

**Run order:** imports → **load data** (builds `samples_for_actions`) → **definitions** (retrieval + pipeline; edit `_PIPELINE_*` flags there to control how much RAG text / full prompt is printed) → **driver** cells (preload → configuration → execute).

## Retrieval improvements (what runs before the LLM)

Retrieval is a multi-stage ranking pipeline:

- **Multi-query support**: `QUERY_STRATEGY` can be a single strategy or a list of strategies.
- **Per-query retrieval**: fetch up to **100** nearest child chunks per query (after index oversample + balance), then MMR/rerank on **100** candidates.
- **Merge + dedupe**: merge candidates and remove duplicates (prefers `(parent_id, child_index)` identity).
- **MMR**: diversify results to reduce redundant chunks while keeping relevance.
- **Reranking**: `rerank_mode` can be `none`, `bm25`, `crossencoder`, `colbert`, or `crossencoder_colbert`. With **`RUN_RERANK_ABLATION = True`** (configuration driver cell), compare modes and save `rerank_comparison_sample_*.json`.
- **Child → parent expansion**: expand only the top-ranked child chunks into parent documents.
- **Final context**: pass the **top 10** ranked parent sections (full text) into the LLM prompt.
- **Wider pool for retrieval metrics**: the rerank ablation persists a **30-chunk reranked candidate pool** per mode under `modes.*.rag_info.candidate_pool` (in addition to the 10 shown to the LLM under `top_results`). This gives `Score.ipynb` a denominator larger than *k* so Recall@10 / nDCG@10 can take fractional values (see `_RERANK_CANDIDATE_POOL_SIZE`).

**Shared helpers:** `utils/rag_utils.py` (FAISS load/save, prediction/config loaders, etc.). Retrieval + action planning logic lives in this notebook.

## Evaluation goals (rerank / retrieval metrics)

**Decision:** optimize **both** (they answer different questions).

1. **Downstream action-plan quality** — primary product goal: grounded JSON plans (threat level, priorities, actions, `knowledge_sources_used`) that match operational needs. Prefer human rubric, LLM-as-judge, or schema/citation checks over keyword overlap on chunks alone.
2. **Query–chunk relevance** — leading indicator: same embedder as the FAISS index, e.g. mean cosine(query, top-*k* chunk texts) per `rerank_mode`. Helps diagnose retrieval without replacing end-to-end evaluation.

The rerank ablation reports persist both the LLM-visible top-10 (`top_results`) and a wider 30-chunk reranked pool (`candidate_pool`). `Score.ipynb` consumes the pool to compute Recall@10 / nDCG@10 (with adaptive graded-relevance thresholds) plus a CRAG-style truthfulness proxy and BERTScore-F1 against a label-derived reference text.

Use `rag_context_rerank_keyword_scores.csv` only as a **diagnostic** (label/action/tier word overlap), not as the main leaderboard. Optional tooling: `utils/rerank_metrics.py` and the **Rerank report metrics** cell at the end of this notebook.

## Inline Parameter Annotations (Plan)

These are the highest-impact parameters for retrieval, reranking, and action-plan output quality.

- `VECTOR_STORE_DIR` / `PREDICTIONS_DIR` / `RESULTS_DIR`: I/O contract for loading FAISS + predictions and saving action plans.
- `_INDEX_EMBED_MODEL` from manifest: keeps retrieval diagnostics tied to the exact embedding model used to build the index.
- `_PER_QUERY_RETRIEVE_K`: candidate width per query before reranking; larger improves recall but increases latency.
- `_DEFAULT_MMR_K`: pool size considered by MMR; controls redundancy suppression strength.
- `_DEFAULT_RERANK_K`: number of candidates scored by rerankers; larger improves ranking headroom with higher compute.
- `_DEFAULT_FINAL_SECTIONS`: final context budget passed to LLM; more sections add recall but can dilute prompt focus.
- `_RERANK_CANDIDATE_POOL_SIZE`: wider reranked pool (default 30) persisted per mode in `rerank_comparison_*.json` under `rag_info.candidate_pool`. The LLM prompt still sees only the top `_DEFAULT_FINAL_SECTIONS` (10); the wider pool is used by `Score.ipynb` as the denominator for Recall@k / nDCG@k.
- `oversample_factor` in `retrieve_child_chunks_for_query(...)`: initial FAISS overfetch multiplier to improve post-filter quality.
- `balance_by_source_file`: prevents one document source from dominating retrieved context.
- `rerank_mode` (`none`, `bm25`, `crossencoder`, `colbert`, `crossencoder_colbert`): ranking strategy choice; primary quality/latency tradeoff.
- `RUN_RERANK_ABLATION`: when enabled, executes all rerank modes and writes comparison reports for offline analysis.
- `ensure_msvc_for_torch_jit()` + `apply_colbert_segmented_maxsim_patch()`: Windows-specific ColBERT compile safeguards.
- LLM prompt flags (`_PIPELINE_*` print/debug controls): tune observability and output verbosity while validating retrieval behavior.
- `DEFAULT_EXPORT_RAG_TOP_N` (via utils): cap for serialized context in output JSON; affects report size and auditability.


In [1]:
import os
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from openai import OpenAI

from utils.project_env import load_project_environment

load_project_environment()

from utils.rag_utils import (
    balance_vector_hits_by_source_file,
    get_dominant_party_info,
    load_attack_and_agentic,
    load_parent_store,
    load_predictions,
    load_vector_store,
    read_manifest,
    save_action_plan,
    save_rerank_comparison_report,
)

VECTOR_STORE_DIR = Path("RAG_docs/vector_store")
PREDICTIONS_DIR = Path("RAG_docs/predictions")
RESULTS_DIR = Path("RAG_docs/action_plans")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# From FAISS manifest; persisted on rerank comparison JSON for offline metrics.
_INDEX_EMBED_MODEL = read_manifest(VECTOR_STORE_DIR).get("embed_model")

In [2]:
vector_store = load_vector_store(VECTOR_STORE_DIR)
predictions_data = load_predictions(PREDICTIONS_DIR, verbose=True)
attack_actions_data, agentic_features_data = load_attack_and_agentic(verbose=True)

n_attack = (
    len(attack_actions_data.get("attacks", {}))
    if isinstance(attack_actions_data, dict)
    else 0
)
has_agentic = isinstance(agentic_features_data, dict) and any(
    t in agentic_features_data for t in ("RAN", "Edge", "Core")
)
print("-" * 60)
print(
    f"Predictions: {len(predictions_data)} | "
    f"Vector store: {'loaded' if vector_store is not None else 'missing'}"
)
print(f"Attack types in attack_options: {n_attack}")
print(f"Agentic features config: {'yes' if has_agentic else 'no'}")
print("-" * 60)


def _include_sample_for_action_plan(sample: Dict[str, Any]) -> bool:
    """Include rows with a non-empty predicted label (including BENIGN)."""
    pl = str(sample.get("predicted_label") or "").strip().upper()
    return bool(pl)


samples_for_actions = [s for s in predictions_data if _include_sample_for_action_plan(s)]
n_empty = len(predictions_data) - len(samples_for_actions)
print(
    f"Samples queued for action planning: {len(samples_for_actions)}"
    + (f" ({n_empty} row(s) with empty predicted_label skipped)." if n_empty else ".")
)


C:\Projects\AML\VFL-RAG\utils\rag_utils.py:213: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return SentenceTransformerEmbeddings(


Found 1 prediction file(s) in RAG_docs\predictions
  Loading: sample_predictions.json
    Loaded 9 prediction(s) from sample_predictions.json
Total: 9 prediction(s)
Loaded attack actions from attack_options.json
Loaded agentic features from agentic_features.json
------------------------------------------------------------
Predictions: 9 | Vector store: loaded
Attack types in attack_options: 9
Agentic features config: no
------------------------------------------------------------
Samples queued for action planning: 9.


In [3]:
# RAG retrieval + LLM query/prompts/pipeline (search lives here next to query construction)

_RAG_PARENTS = load_parent_store(VECTOR_STORE_DIR)


_PER_QUERY_RETRIEVE_K = 100

# Ranking width vs LLM context:
#   - retrieve/MMR/rerank on up to ``_DEFAULT_RERANK_K`` candidates (100);
#   - ``_DEFAULT_FINAL_SECTIONS`` sections (10) go to the LLM prompt;
#   - ``_RERANK_CANDIDATE_POOL_SIZE`` sections (30) are persisted per mode in
#     ``rerank_comparison_*.json`` under ``rag_info.candidate_pool`` so that
#     ``Score.ipynb`` can compute Recall@k / nDCG@k against a denominator > k.
_DEFAULT_FINAL_SECTIONS = 10
_DEFAULT_MMR_K = 100
_DEFAULT_RERANK_K = 100
_RERANK_CANDIDATE_POOL_SIZE = 30


def _chunk_key(d: Dict[str, Any]) -> Tuple[Any, ...]:
    """Stable identity key for a child chunk."""
    pid = d.get("parent_id")
    cix = d.get("child_index")
    if pid is not None and cix is not None:
        return ("pc", str(pid), int(cix))
    return (
        "legacy",
        str(d.get("source_file") or "").strip(),
        str(d.get("title") or "").strip(),
        str(d.get("chunk_text") or "").strip()[:120],
    )


def _as_score_from_dist(dist: Any) -> float:
    d = float(dist) if dist is not None else 0.0
    return 1.0 / (1.0 + max(d, 0.0))


def retrieve_child_chunks_for_query(
    vector_store: Any,
    query: str,
    *,
    top_k: int = _PER_QUERY_RETRIEVE_K,
    oversample_factor: int = 6,
    balance_by_source_file: bool = True,
) -> List[Dict[str, Any]]:
    """Retrieve child chunks only (NO parent expansion)."""
    if not vector_store or not query:
        return []

    fetch_k = min(300, max(int(top_k), int(top_k) * int(oversample_factor)))
    vector_results = vector_store.similarity_search_with_score(query, k=fetch_k)

    if balance_by_source_file:
        vector_results = balance_vector_hits_by_source_file(vector_results, int(top_k))
    else:
        vector_results = vector_results[: int(top_k)]

    out: List[Dict[str, Any]] = []
    for doc, dist in vector_results:
        meta = doc.metadata or {}
        title = meta.get("retrieval_title") or meta.get("title", "Unknown")
        src = meta.get("source_file", "")
        pid = meta.get("parent_id")
        cix = meta.get("child_index")
        chunk_text = doc.page_content or meta.get("text", "") or ""

        out.append(
            {
                "title": title,
                "source_file": src,
                "parent_id": pid,
                "child_index": cix,
                "chunk_text": chunk_text,
                "vector_score": _as_score_from_dist(dist),
            }
        )

    return out


def merge_and_dedupe_child_chunks(
    results_by_query: List[List[Dict[str, Any]]],
) -> List[Dict[str, Any]]:
    """Merge child-chunk sets; dedupe by (parent_id, child_index) when available."""
    best: Dict[Tuple[Any, ...], Dict[str, Any]] = {}

    for results in results_by_query:
        for d in results:
            k = _chunk_key(d)
            prev = best.get(k)
            if not prev or float(d.get("vector_score", 0.0) or 0.0) > float(prev.get("vector_score", 0.0) or 0.0):
                best[k] = d

    merged = list(best.values())
    merged.sort(key=lambda x: float(x.get("vector_score", 0.0) or 0.0), reverse=True)
    return merged


def _cosine(a: List[float], b: List[float]) -> float:
    import math

    if not a or not b or len(a) != len(b):
        return 0.0
    da = sum(x * x for x in a)
    db = sum(x * x for x in b)
    if da <= 0.0 or db <= 0.0:
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    return float(dot) / float(math.sqrt(da) * math.sqrt(db))


def _embed_query(vector_store: Any, query: str) -> List[float]:
    # LangChain FAISS typically exposes `embedding_function`.
    ef = getattr(vector_store, "embedding_function", None)
    if ef is None:
        raise ValueError("vector_store.embedding_function missing; cannot run MMR")
    if hasattr(ef, "embed_query"):
        return list(ef.embed_query(query))
    # Fallback for older wrappers
    if callable(ef):
        return list(ef(query))
    raise ValueError("Unsupported embedding function")


def _embed_texts(vector_store: Any, texts: List[str]) -> List[List[float]]:
    ef = getattr(vector_store, "embedding_function", None)
    if ef is None:
        raise ValueError("vector_store.embedding_function missing; cannot run MMR")
    if hasattr(ef, "embed_documents"):
        return [list(v) for v in ef.embed_documents(texts)]
    # Slow fallback: embed one-by-one via embed_query
    if hasattr(ef, "embed_query"):
        return [list(ef.embed_query(t)) for t in texts]
    raise ValueError("Unsupported embedding function")


def mmr_select(
    vector_store: Any,
    query: str,
    candidates: List[Dict[str, Any]],
    *,
    k: int = _DEFAULT_MMR_K,
    lambda_mult: float = 0.5,
) -> List[Dict[str, Any]]:
    """MMR over candidate chunk embeddings."""
    if not candidates:
        return []

    k = max(1, min(int(k), len(candidates)))

    texts = [str(c.get("chunk_text") or "") for c in candidates]
    qv = _embed_query(vector_store, query)
    dvs = _embed_texts(vector_store, texts)

    # Precompute query similarities
    sim_q = [_cosine(qv, dv) for dv in dvs]

    selected: List[int] = []
    remaining = set(range(len(candidates)))

    # Start from best by query similarity
    first = max(remaining, key=lambda i: sim_q[i])
    selected.append(first)
    remaining.remove(first)

    while remaining and len(selected) < k:
        def mmr_score(i: int) -> float:
            # max similarity to already-selected docs
            max_sim = max(_cosine(dvs[i], dvs[j]) for j in selected) if selected else 0.0
            return float(lambda_mult) * float(sim_q[i]) - (1.0 - float(lambda_mult)) * float(max_sim)

        nxt = max(remaining, key=mmr_score)
        selected.append(nxt)
        remaining.remove(nxt)

    return [candidates[i] for i in selected]


_CROSSENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
_COLBERT_MODEL_NAME = "colbert-ir/colbertv2.0"

_CROSS_ENCODER = None
_COLBERT = None
_RERANKERS_CONFIRMED = False


def ensure_rerankers_loaded(
    *,
    crossencoder_model: str = _CROSSENCODER_MODEL_NAME,
    colbert_model: str = _COLBERT_MODEL_NAME,
) -> Tuple[Any, Any]:
    """Mandatory reranker loader.

    Raises a clear error if dependencies/models are missing.
    """
    global _CROSS_ENCODER, _COLBERT, _RERANKERS_CONFIRMED

    if _CROSS_ENCODER is None:
        try:
            from sentence_transformers import CrossEncoder  # type: ignore
        except Exception as e:
            raise ImportError(
                "CrossEncoder reranker is REQUIRED. Install: pip install sentence-transformers"
            ) from e
        from utils.compute_device import cross_encoder_device_string

        _CROSS_ENCODER = CrossEncoder(
            crossencoder_model,
            device=cross_encoder_device_string(),
        )

    if _COLBERT is None:
        from utils.msvc_jit_env import ensure_msvc_for_torch_jit

        ensure_msvc_for_torch_jit()
        from utils.colbert_windows_cpp import apply_colbert_segmented_maxsim_patch

        apply_colbert_segmented_maxsim_patch()
        try:
            import utils.langchain_ragatouille_shim  # noqa: F401 — LC 1.x vs RAGatouille import paths
            from ragatouille import RAGPretrainedModel  # type: ignore
        except Exception as e:
            raise ImportError(
                "ColBERT reranker is REQUIRED. Install: pip install ragatouille"
            ) from e
        _COLBERT = RAGPretrainedModel.from_pretrained(colbert_model)

    if not _RERANKERS_CONFIRMED:
        print(
            "Rerankers loaded OK: "
            f"CrossEncoder='{crossencoder_model}', ColBERT='{colbert_model}'"
        )
        _RERANKERS_CONFIRMED = True

    return (_CROSS_ENCODER, _COLBERT)


def crossencoder_rerank(
    query: str,
    candidates: List[Dict[str, Any]],
) -> List[float]:
    """Cross-encoder rerank scores; mandatory (no fallback)."""
    ce, _ = ensure_rerankers_loaded()
    pairs = [(query, str(c.get("chunk_text") or "")) for c in candidates]
    scores = ce.predict(pairs)
    return [float(s) for s in scores]


def colbert_rerank(
    query: str,
    candidates: List[Dict[str, Any]],
) -> List[float]:
    """ColBERT rerank scores; mandatory (no fallback)."""
    _, cb = ensure_rerankers_loaded()
    docs = [str(c.get("chunk_text") or "") for c in candidates]
    scored = cb.rerank(query=query, documents=docs, k=len(docs))

    # Keep ordering; ragatouille returns dicts containing the raw `document` string.
    score_map = {str(x.get("document")): float(x.get("score", 0.0) or 0.0) for x in scored}
    return [float(score_map.get(d, 0.0)) for d in docs]


import re as _re_bm25  # used below for BM25 tokenization only


def _bm25_tokenize(text: str) -> List[str]:
    """Lowercase word-char tokenizer used for BM25 scoring (corpus + query)."""
    return [t for t in _re_bm25.findall(r"[A-Za-z0-9]+", str(text or "").lower()) if t]


def bm25_rerank(
    query: str,
    candidates: List[Dict[str, Any]],
) -> List[float]:
    """Okapi BM25 rerank scores (lexical, sparse).

    Implements the classic BM25 reranker from the BEIR benchmark (Thakur et al., 2021,
    arXiv:2104.08663). Scores each candidate's ``chunk_text`` against the anchor query
    using ``rank_bm25.BM25Okapi`` with default ``k1`` and ``b`` parameters.
    """
    try:
        from rank_bm25 import BM25Okapi  # type: ignore
    except Exception as e:  # pragma: no cover — mirrors CrossEncoder/ColBERT behavior
        raise ImportError(
            "BM25 reranker is REQUIRED. Install: pip install rank-bm25"
        ) from e

    corpus_tokens = [_bm25_tokenize(c.get("chunk_text") or "") for c in candidates]
    # Guard against fully-empty corpora (BM25Okapi divides by avgdl).
    if not corpus_tokens or all(len(toks) == 0 for toks in corpus_tokens):
        return [0.0 for _ in candidates]

    bm25 = BM25Okapi(corpus_tokens)
    q_tokens = _bm25_tokenize(query)
    if not q_tokens:
        return [0.0 for _ in candidates]
    scores = bm25.get_scores(q_tokens)
    return [float(s) for s in scores]


def rerank_crossencoder_plus_colbert(
    query: str,
    candidates: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Attach rerank scores and sort descending."""
    ce = crossencoder_rerank(query, candidates)
    cb = colbert_rerank(query, candidates)

    out: List[Dict[str, Any]] = []
    for c, s_ce, s_cb in zip(candidates, ce, cb):
        d = dict(c)
        d["crossencoder_score"] = float(s_ce)
        d["colbert_score"] = float(s_cb)
        # Simple combination; you can tune weights later.
        d["rerank_score"] = float(s_ce) + float(s_cb)
        out.append(d)

    out.sort(key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True)
    return out


def expand_parent_sections(
    ranked_children: List[Dict[str, Any]],
    *,
    top_sections: int = _DEFAULT_FINAL_SECTIONS,
    max_parent_chars: int = 12000,
) -> List[Dict[str, Any]]:
    """Expand top-ranked child chunks into parent contextual sections.

    Removes duplicates by keeping the most relevant section when the same
    (parent_id) or (source_file,title) repeats.
    """
    if not ranked_children:
        return []

    # Build candidate sections first (may include duplicates)
    candidates: List[Dict[str, Any]] = []

    for c in ranked_children:
        score = float(c.get("rerank_score", c.get("vector_score", 0.0) or 0.0))
        pid = c.get("parent_id")

        if pid is None:
            candidates.append(
                {
                    "title": c.get("title", "Unknown"),
                    "source_file": c.get("source_file", ""),
                    "text": str(c.get("chunk_text") or ""),
                    "score": score,
                    "child_index": c.get("child_index"),
                }
            )
            continue

        pid_s = str(pid)
        parent = _RAG_PARENTS.get(pid_s) or _RAG_PARENTS.get(pid) or {}
        ptext = str(parent.get("text") or "")
        if max_parent_chars and len(ptext) > int(max_parent_chars):
            ptext = ptext[: int(max_parent_chars)] + "\n\n[parent truncated]"

        title = parent.get("retrieval_title") or c.get("title") or "Unknown"
        candidates.append(
            {
                "title": title,
                "source_file": c.get("source_file", ""),
                "text": f"{title}\n\n{ptext}" if ptext else str(c.get("chunk_text") or ""),
                "score": score,
                "parent_id": pid_s,
                "child_index": c.get("child_index"),
            }
        )

    # Dedupe: prefer parent_id when present, otherwise (source_file,title)
    best: Dict[Tuple[Any, ...], Dict[str, Any]] = {}
    for s in candidates:
        if s.get("parent_id"):
            k: Tuple[Any, ...] = ("parent", str(s.get("parent_id")))
        else:
            k = ("doc", str(s.get("source_file") or "").strip(), str(s.get("title") or "").strip())

        prev = best.get(k)
        if not prev or float(s.get("score", 0.0) or 0.0) > float(prev.get("score", 0.0) or 0.0):
            best[k] = s

    sections = list(best.values())
    sections.sort(key=lambda x: float(x.get("score", 0.0) or 0.0), reverse=True)
    return sections[: int(top_sections)]


def retrieve_rag_context_multi(
    vector_store: Any,
    queries: List[str],
    *,
    top_k: int = _DEFAULT_FINAL_SECTIONS,
    oversample_factor: int = 6,
    balance_by_source_file: bool = True,
    mmr_k: int = _DEFAULT_MMR_K,
    rerank_k: int = _DEFAULT_RERANK_K,
    lambda_mult: float = 0.5,
    max_parent_chars: int = 12000,
    rerank_mode: str = "crossencoder_colbert",
) -> Tuple[List[Dict[str, Any]], List[List[Dict[str, Any]]]]:
    """Merge + dedupe → MMR → rerank (see ``rerank_mode``) → parent expansion → top sections.

    ``rerank_mode``: ``none`` | ``bm25`` | ``crossencoder`` | ``colbert`` | ``crossencoder_colbert``.
    """
    qs = [" ".join((q or "").split()) for q in (queries or [])]
    qs = [q for q in qs if q]
    if not qs:
        return ([], [])

    per_query: List[List[Dict[str, Any]]] = []
    for q in qs:
        per_query.append(
            retrieve_child_chunks_for_query(
                vector_store,
                q,
                top_k=int(_PER_QUERY_RETRIEVE_K),
                oversample_factor=oversample_factor,
                balance_by_source_file=balance_by_source_file,
            )
        )

    merged = merge_and_dedupe_child_chunks(per_query)

    anchor_query = qs[0]

    mmr_pool = mmr_select(
        vector_store,
        anchor_query,
        merged,
        k=int(mmr_k),
        lambda_mult=float(lambda_mult),
    )

    rm = (rerank_mode or "crossencoder_colbert").strip().lower()
    if rm == "none":
        reranked: List[Dict[str, Any]] = []
        for c in mmr_pool:
            d = dict(c)
            vs = float(d.get("vector_score", 0.0) or 0.0)
            d["rerank_score"] = vs
            d["ranking_method"] = "vector_only"
            reranked.append(d)
        reranked.sort(
            key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True
        )
    elif rm == "bm25":
        bm25_scores = bm25_rerank(anchor_query, mmr_pool)
        reranked = []
        for c, s_bm in zip(mmr_pool, bm25_scores):
            d = dict(c)
            d["bm25_score"] = float(s_bm)
            d["rerank_score"] = float(s_bm)
            d["ranking_method"] = "bm25"
            reranked.append(d)
        reranked.sort(
            key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True
        )
    elif rm == "crossencoder":
        ce_scores = crossencoder_rerank(anchor_query, mmr_pool)
        reranked = []
        for c, s_ce in zip(mmr_pool, ce_scores):
            d = dict(c)
            d["crossencoder_score"] = float(s_ce)
            d["rerank_score"] = float(s_ce)
            d["ranking_method"] = "crossencoder"
            reranked.append(d)
        reranked.sort(
            key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True
        )
    elif rm == "colbert":
        cb_scores = colbert_rerank(anchor_query, mmr_pool)
        reranked = []
        for c, s_cb in zip(mmr_pool, cb_scores):
            d = dict(c)
            d["colbert_score"] = float(s_cb)
            d["rerank_score"] = float(s_cb)
            d["ranking_method"] = "colbert"
            reranked.append(d)
        reranked.sort(
            key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True
        )
    else:
        reranked = rerank_crossencoder_plus_colbert(anchor_query, mmr_pool)
        for d in reranked:
            d["ranking_method"] = "crossencoder_colbert"

    top_children = reranked[: int(rerank_k)]
    final_sections = expand_parent_sections(
        top_children,
        top_sections=int(top_k),
        max_parent_chars=int(max_parent_chars),
    )

    final_sections.sort(key=lambda x: float(x.get("score", 0.0) or 0.0), reverse=True)
    return (final_sections[: int(top_k)], per_query)


def summarize_rag_docs(
    docs: List[Dict[str, Any]],
    *,
    max_docs: int = 5,
    snippet_chars: int = 220,
) -> List[Dict[str, Any]]:
    """Compact summary objects for printing/debug.

    Supports both shapes:
    - final sections: {title, text, score, source_file}
    - child chunks:   {title, chunk_text, vector_score, source_file}
    """
    out: List[Dict[str, Any]] = []
    for d in (docs or [])[: int(max_docs)]:
        txt = str(d.get("text") or d.get("chunk_text") or "")
        snippet = txt[: int(snippet_chars)]
        if len(txt) > int(snippet_chars):
            snippet += "..."

        score = d.get("score")
        if score is None:
            score = d.get("rerank_score")
        if score is None:
            score = d.get("vector_score")

        out.append(
            {
                "title": d.get("title", "Unknown"),
                "source_file": d.get("source_file", ""),
                "score": float(score or 0.0),
                "snippet": " ".join(snippet.split()),
            }
        )
    return out


def _top_features_for_tier(
    sample: Dict[str, Any],
    tier: str,
    *,
    top_n: int = 3,
    score_key: str = "pct_contribution",
) -> List[str]:
    """Deterministic top-N feature names for a tier (RAN/Edge/Core)."""
    shap_expl = sample.get("shap_explanation", {}) or {}
    feat_contribs = shap_expl.get("feature_contributions", {}) or {}
    feats = feat_contribs.get(tier, {}) or {}

    scored: List[Tuple[str, float]] = []
    if isinstance(feats, dict):
        for feat_name, meta in feats.items():
            if not isinstance(meta, dict):
                continue
            score = float(meta.get(score_key, 0.0) or 0.0)
            if score > 0.0:
                scored.append((feat_name, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [name for name, _ in scored[:top_n]]


def extract_sample_summary(sample: Dict[str, Any]) -> Dict[str, Any]:
    """Normalize prediction fields used by query builders (single source of truth)."""
    label = (sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN").strip().upper()
    confidence = float(sample.get("confidence", 0.0) or 0.0)

    shap_expl = sample.get("shap_explanation", {}) or {}
    dominant_agent = (shap_expl.get("dominant_agent") or "").strip()
    dominant_tier = dominant_agent if dominant_agent in ("RAN", "Edge", "Core") else "Unknown"
    dominant_pct = float(shap_expl.get("dominant_contribution_pct", 0.0) or 0.0) * 100.0

    party_pct = shap_expl.get("party_contributions_pct", {}) or {}
    tier_contributions_pct = {
        "RAN": float(party_pct.get("RAN", 0.0) or 0.0) * 100.0,
        "Edge": float(party_pct.get("Edge", 0.0) or 0.0) * 100.0,
        "Core": float(party_pct.get("Core", 0.0) or 0.0) * 100.0,
    }

    top_features = {
        "RAN": _top_features_for_tier(sample, "RAN", top_n=3),
        "Edge": _top_features_for_tier(sample, "Edge", top_n=3),
        "Core": _top_features_for_tier(sample, "Core", top_n=3),
    }

    return {
        "label": label,
        "confidence": confidence,
        "dominant_tier": dominant_tier,
        "dominant_pct": dominant_pct,
        "tier_contributions_pct": tier_contributions_pct,
        "top_features": top_features,
    }


_TEMPLATE_QUERY = (
    "Find mitigation actions, detection steps, and response playbooks for a {label} attack in a telecom network. "
    "Use explainability evidence from all tiers and weight recommendations by contribution strength. "
    "Tier contributions: RAN {ran_pct:.0f}%, Edge {edge_pct:.0f}%, Core {core_pct:.0f}%. "
    "Top indicators per tier: RAN: {ran_feats}. Edge: {edge_feats}. Core: {core_feats}. "
    "Focus on controls appropriate for confidence {confidence:.0%}."
)


def build_template_rag_query(sample: Dict[str, Any]) -> str:
    """Deterministic template query used for vector retrieval (NO LLM).

    Query options supported in this notebook:
    1) Template-based (deterministic)            -> `build_template_rag_query(sample)`
    2) "BERT-based" style rephrase (optional)     -> `rephrase_template_query_deterministic(template_query)`
       (Note: implemented as deterministic rules; no model downloads.)
    3) LLM-based query variants (optional)       -> `build_llm_rag_queries(sample)`
       (Returns two variants derived from the template + top features in prediction JSON.)

    For RAG retrieval, we intentionally use **only option (1)** for determinism.
    """
    s = extract_sample_summary(sample)

    def fmt(feats: List[str]) -> str:
        return ", ".join(feats) if feats else "no strong features"

    tier_pct = s.get("tier_contributions_pct", {}) or {}

    return _TEMPLATE_QUERY.format(
        label=s["label"],
        confidence=s["confidence"],
        ran_pct=float(tier_pct.get("RAN", 0.0) or 0.0),
        edge_pct=float(tier_pct.get("Edge", 0.0) or 0.0),
        core_pct=float(tier_pct.get("Core", 0.0) or 0.0),
        ran_feats=fmt(s["top_features"].get("RAN", [])),
        edge_feats=fmt(s["top_features"].get("Edge", [])),
        core_feats=fmt(s["top_features"].get("Core", [])),
    )


def build_llm_rag_queries(sample: Dict[str, Any]) -> Tuple[str, str]:
    """Optional: return (concise_query, expanded_query) using an LLM.

    Not used for retrieval by default.
    """
    base = build_template_rag_query(sample)
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return (base, base)

    client = OpenAI(api_key=api_key)
    prompt = (
        "Rewrite the following retrieval query into two variants for vector search.\n\n"
        "Variant A: concise (1 sentence, keep feature tokens).\n"
        "Variant B: expanded (2-3 sentences, add synonyms, keep feature tokens).\n\n"
        "Return JSON only: {\"concise\": \"...\", \"expanded\": \"...\"}.\n\n"
        f"QUERY:\n{base}"
    )

    resp = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "Return only valid JSON."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=250,
    )

    txt = (resp.choices[0].message.content or "").strip()
    try:
        start = txt.find("{")
        end = txt.rfind("}") + 1
        obj = json.loads(txt[start:end]) if start >= 0 and end > start else {}
        concise = str(obj.get("concise") or base).strip()
        expanded = str(obj.get("expanded") or base).strip()
        return (concise, expanded)
    except Exception:
        return (base, base)


def rephrase_template_query_deterministic(template_query: str) -> str:
    """Deterministic query rephrase (no models / no downloads).

    Not used for retrieval by default.
    """
    q = (template_query or "").strip()
    if not q:
        return q

    # Simple, stable synonym swaps + light restructuring.
    swaps = [
        ("Find mitigation actions, detection steps, and response playbooks for a ", "Retrieve mitigation and detection guidance for "),
        ("telecom network", "telecommunications network"),
        ("Prioritize the ", "Focus on the "),
        ("Use these top indicators as keywords:", "Key indicators:"),
        ("Focus on controls appropriate for confidence", "Tailor controls to confidence"),
    ]

    for a, b in swaps:
        q = q.replace(a, b)

    # Ensure a compact single-line output for embedding/search.
    q = " ".join(q.split())
    return q


def create_prompt(
    sample_data: Dict[str, Any],
    rag_results: List[Dict[str, Any]],
    attack_actions_data: Optional[Dict[str, Any]] = None,
    agentic_features_data: Optional[Dict[str, Any]] = None,
) -> str:
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample_data)

    # IMPORTANT: Do NOT pass the full prediction/SHAP payload to the LLM.
    # Only pass compact evidence: all tier contributions + top-3 features per tier.
    s = extract_sample_summary(sample_data)
    condensed_evidence = {
        "predicted_label": str(s.get("label") or predicted_label),
        "confidence": float(s.get("confidence", confidence) or 0.0),
        "dominant_tier": s.get("dominant_tier", dominant_tier),
        "dominant_contribution_pct": float(s.get("dominant_pct", dominant_pct) or 0.0),
        "tier_contributions_pct": s.get("tier_contributions_pct", {}),
        "top_features_by_tier": s.get("top_features", {}),
    }

    attack_actions_context = ""
    if attack_actions_data and "attacks" in attack_actions_data:
        attack_type = predicted_label.upper()
        if attack_type in attack_actions_data["attacks"]:
            recommended_actions = attack_actions_data["attacks"][attack_type]
            attack_actions_context = (
                f"\n\nAttack-Specific Recommended Actions (from attack_options.json):\n"
                f"For {predicted_label} attack, recommended actions: {', '.join(recommended_actions)}\n"
            )
        else:
            attack_actions_context = (
                f"\n\nAttack-Specific Actions: No specific recommendations for {predicted_label}.\n"
            )

    agentic_context = ""
    if agentic_features_data:
        agentic_context = (
            "\n\nAgentic Features and Actions by Network Tier (from agentic_features.json):\n"
            "Use evidence from all tiers (contributions + top features) to set action priority and sequencing.\n"
        )
        tf = condensed_evidence.get("top_features_by_tier", {}) or {}
        tier_pct = condensed_evidence.get("tier_contributions_pct", {}) or {}
        for tier in ["RAN", "Edge", "Core"]:
            if tier in agentic_features_data:
                tier_data = agentic_features_data[tier]
                actions = tier_data.get("actions", [])
                tier_feats = tf.get(tier, []) if isinstance(tf, dict) else []
                pct = float(tier_pct.get(tier, 0.0) or 0.0)
                agentic_context += f"\n{tier} Network Tier:\n"
                agentic_context += f"  - Tier contribution: {pct:.1f}%\n"
                agentic_context += f"  - Top evidence features (top 3): {', '.join(tier_feats) if tier_feats else 'none'}\n"
                agentic_context += f"  - Allowed tier actions: {', '.join(actions)}\n"

    # RAG context: shared helper so the notebook can print the same text the model sees.
    rag_context = build_rag_context_text(rag_results)

    network_tier_info = ""
    if dominant_tier:
        network_tier_info = f"\n- Dominant network tier: {dominant_tier} (contribution: {dominant_pct:.1f}%)"
        if dominant_party:
            network_tier_info += f"\n- Dominant party: {dominant_party}"

    return f"""You are a cybersecurity decision-making agent specialized in attack response orchestration.
        Your role is NOT to invent mitigations, but to SELECT and ASSIGN actions from a predefined
        action set using explainability signals and agentic features.

        =====================
        INPUT CONTEXT
        =====================

        Prediction summary:
        - sample_id: {sample_id}
        - predicted_label: {predicted_label}
        - confidence: {confidence:.1%}

        Explainability & agentic evidence:
        - Party-level contributions and dominance:
        {network_tier_info}

        - Condensed feature evidence (top 3 per tier):
        {json.dumps(condensed_evidence, indent=2)}

        Allowed actions (STRICT CONSTRAINT):
        {attack_actions_context}
        • Only actions listed above are allowed.
        • Do NOT invent, rename, generalize, or merge actions.

        Agentic decision signals:
        {agentic_context}

        Retrieved knowledge (RAG – optional support, may be empty):
        {rag_context}

        =====================
        TASK
        =====================

        Using ONLY the information above, generate an agent-ready action plan that:

        1) Interprets how {predicted_label} manifests across network tiers (RAN, Edge, Core).
        2) Identifies which evidence party MUST trigger mitigation first.
        3) Selects actions ONLY from the provided Allowed actions list.
        4) Assigns each action to the MOST appropriate executing party and network tier.
        5) Provides explicit, evidence-grounded reasoning for EACH action.
        6) Adapts aggressiveness and execution priority based on confidence ({confidence:.1%}).
        7) If no allowed action is suitable, return empty action lists.

        =====================
        OUTPUT FORMAT (STRICT)
        =====================

        Return a VALID JSON object ONLY:

        {{
          "threat_level": "Critical|High|Medium|Low",

          "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
          ],

          "primary_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken ",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],

          "supporting_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "why this action supports or complements the primary action"
            }}
          ],
          "overall_reasoning": "Concise summary explaining party prioritization, tier ordering, and action selection logic",
          "execution_priority": "Immediate|High|Standard|Low",
          "knowledge_sources_used": [
            "allowed_actions_context",
            "attack_actions_context"
          ]
        }}

        =====================
        HARD RULES (DO NOT VIOLATE)
        =====================

        - Do NOT output text outside the JSON.
        - Do NOT generate actions not listed in Allowed actions.
        - The "all_actions" list MUST be the union of primary_actions and supporting_actions.
        - Do NOT alter action or party names.
        - Every action MUST include explicit reasoning tied to evidence or agentic rules.
        - Use all tier evidence; dominant tier may lead primary actions, but include supporting actions for other contributing tiers.
        - If RAG context is empty, rely ONLY on explainability and agentic context.
        """


def create_prompt_without_RAG(sample_data: Dict[str, Any]) -> str:
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)

    return f"""You are a cybersecurity expert responsible for selecting mitigation actions
      based on attack predictions and domain knowledge.

      Your objective is to decide WHAT actions should be taken and WHY,
      strictly using a predefined set of allowed actions.

      =====================
      INPUT
      =====================

      Prediction summary:
      - sample_id: {sample_id}
      - predicted_label: {predicted_label}
      - confidence: {confidence:.1%}

      =====================
      TASK
      =====================

      Using ONLY the information above:

      1) Assess the severity of the predicted attack ({predicted_label})
        and assign an appropriate threat level.
      2) Select all relevant mitigation actions within 2 or 3 words
      3) Ensure selected actions are appropriate for the predicted attack type.
      4) Adapt action selection and urgency based on confidence ({confidence:.1%}).
      5) If no action is applicable, return an empty action list.

      =====================
      OUTPUT (STRICT JSON ONLY)
      =====================

      {{
        "threat_level": "Critical|High|Medium|Low",
        "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
        ],
        "primary_actions": [
            {{
              "action": "any action name  to be taken",
              "network_tier": "RAN|Edge|Core",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],
        "overall_reasoning": "Brief explanation describing why these actions were selected and how confidence influenced the decision",
        "execution_priority": "Immediate|High|Standard|Low",
        "knowledge_sources_used": [
          "publicly_available_online_sources"
        ]
      }}

      =====================
      HARD RULES
      =====================

      - Do NOT output text outside the JSON.
      - Do NOT invent, rename, or generalize actions.
      - Use short, standard mitigation action names (2–3 words).
      - If no action applies, return empty lists for `all_actions` and `primary_actions`.
      - Keep reasoning concise and decision-focused.
      """


def call_llm_api(prompt: str) -> Optional[Dict[str, Any]]:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set")
        return None

    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {
                "role": "system",
                "content": "You are a cybersecurity expert. Return only valid JSON.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
    )

    response_text = response.choices[0].message.content.strip()
    try:
        start = response_text.find("{")
        end = response_text.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except Exception as e:
        print(f"Warning: Could not parse JSON: {e}")

    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard",
        "knowledge_sources_used": [],
    }


def _normalize_strategy_to_keys(strategy: Any) -> List[str]:
    """Normalize a strategy value into a list of strategy keys."""
    if isinstance(strategy, list):
        return [str(x).strip().lower() for x in strategy if str(x).strip()]
    s = str(strategy or "").strip().lower()
    return [s] if s else ["template"]


def _vprint(msg: str = "") -> None:
    print(msg)


# --- Notebook observability (stdout when you run the pipeline driver) ---
_PIPELINE_SHOW_UNUSED_QUERY_VARIANTS = False
_PIPELINE_PRINT_PER_QUERY_CHUNKS = True
_PIPELINE_PRINT_MERGED_CHUNK_SUMMARY = True
_PIPELINE_PRINT_RAG_CONTEXT_VERBATIM = True
_PIPELINE_PRINT_FULL_LLM_PROMPT = True
_PIPELINE_PRINT_FULL_PROMPT_EACH_ABLATION_MODE = False
# Compact driver output: attack type, top-N source|title per rerank mode, full LLM JSON.
_PIPELINE_COMPACT_STDOUT = False
_PIPELINE_COMPACT_TOP_SOURCES = 10


def build_rag_context_text(
    rag_results: List[Dict[str, Any]],
    *,
    max_sections: int = 10,
) -> str:
    "Exact Knowledge Base block embedded in the LLM user prompt (for notebook inspection)."
    top = (rag_results or [])[: int(max_sections)]
    if not top:
        return "\n\nKnowledge Base: No relevant documents found from RAG search."
    parts = ["\n\nKnowledge Base Context (from RAG search):\n"]
    for idx, result in enumerate(top, 1):
        title = result.get("title", "Unknown")
        parts.append(f"\n[{idx}] {title}\n")
        parts.append(f"{result.get('text', '')}\n")
    return "".join(parts)


def _print_pipeline_prompt_artifacts(
    rag_results: List[Dict[str, Any]],
    prompt: str,
    *,
    header_extra: str = "",
    ablation_mode_index: Optional[int] = None,
    ablation_modes_total: Optional[int] = None,
) -> None:
    is_ablation = (
        ablation_mode_index is not None
        and ablation_modes_total is not None
        and int(ablation_modes_total) > 1
    )
    print_full = bool(_PIPELINE_PRINT_FULL_LLM_PROMPT)
    if is_ablation and not _PIPELINE_PRINT_FULL_PROMPT_EACH_ABLATION_MODE:
        print_full = print_full and int(ablation_mode_index) == 0

    if _PIPELINE_PRINT_RAG_CONTEXT_VERBATIM:
        _vprint("\n" + "=" * 80)
        _vprint("RAG CONTEXT (verbatim block inside the LLM user message)" + header_extra)
        _vprint("=" * 80)
        _vprint(build_rag_context_text(rag_results))

    if print_full:
        _vprint("\n" + "=" * 80)
        _vprint(f"LLM SYSTEM MESSAGE (API; see call_llm_api){header_extra}")
        _vprint("=" * 80)
        _vprint("You are a cybersecurity expert. Return only valid JSON.")
        _vprint("\n" + "=" * 80)
        _vprint(f"FULL LLM USER PROMPT{header_extra} — {len(prompt)} characters")
        _vprint("=" * 80)
        _vprint(prompt)
    elif is_ablation and _PIPELINE_PRINT_RAG_CONTEXT_VERBATIM:
        _vprint(
            f"\n(Full user prompt omitted for this ablation mode; "
            f"length {len(prompt)} chars. "
            f"Set _PIPELINE_PRINT_FULL_PROMPT_EACH_ABLATION_MODE = True to print every mode.)"
        )


def _format_score_for_print(score: float) -> str:
    s = float(score or 0.0)
    if 0.0 <= s <= 1.0:
        return f"{s:.2%} (cosine-like)"
    return f"{s:.4f} (rank score)"


def _print_doc_summary(title: str, docs: List[Dict[str, Any]]) -> None:
    _vprint(title)
    if not docs:
        _vprint("  (no documents retrieved)")
        return
    for j, d in enumerate(summarize_rag_docs(docs, max_docs=5), 1):
        sc = _format_score_for_print(float(d.get("score", 0.0) or 0.0))
        _vprint(
            f"  [{j}] {d['title']} | {d['source_file']} | {sc}\n"
            f"      {d['snippet']}"
        )


_RERANK_MODE_LABELS: Dict[str, str] = {
    "none": "no rerank (vector + MMR only)",
    "bm25": "BM25 lexical rerank",
    "crossencoder": "CrossEncoder rerank",
    "colbert": "ColBERT rerank",
    "crossencoder_colbert": "CrossEncoder + ColBERT (sum)",
}


def _attack_type_for_display(sample: Dict[str, Any]) -> str:
    return str(
        sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN"
    ).strip().upper()


def _print_compact_top_sources_for_mode(
    mode_key: str,
    rag_results: List[Dict[str, Any]],
    *,
    top_n: int,
) -> None:
    label = _RERANK_MODE_LABELS.get(str(mode_key), str(mode_key))
    print(f"\nTop {int(top_n)} sources — {mode_key} ({label})")
    docs = summarize_rag_docs(
        rag_results or [],
        max_docs=int(top_n),
        snippet_chars=1,
    )
    if not docs:
        print("  (none)")
        return
    for j, d in enumerate(docs, 1):
        sf = str(d.get("source_file") or "").strip() or "(unknown source)"
        title = str(d.get("title") or "").strip() or "(no title)"
        print(f"  [{j}] {sf} | {title}")


def _print_llm_response_compact(llm_response: Optional[Dict[str, Any]]) -> None:
    print("\n--- LLM response (JSON) ---")
    if not llm_response:
        print("Failed to get LLM response")
        return
    print(json.dumps(llm_response, indent=2, ensure_ascii=False))


def _print_llm_response_summary(llm_response: Optional[Dict[str, Any]]) -> None:
    if not llm_response:
        print("Failed to get LLM response")
        return
    print("\n--- LLM response ---")
    print(f"  Threat level: {llm_response.get('threat_level')}")
    print(f"  Execution priority: {llm_response.get('execution_priority')}")
    print(f"  Primary actions: {llm_response.get('primary_actions', [])}")
    reasoning = (llm_response.get("overall_reasoning") or "")[:500]
    if len(llm_response.get("overall_reasoning") or "") > 500:
        reasoning += "..."
    print(f"  Overall reasoning: {reasoning}")


def run_agent_pipeline_for_sample(
    sample: Dict[str, Any],
    vector_store: Any,
    attack_actions_data: Optional[Dict[str, Any]],
    agentic_features_data: Optional[Dict[str, Any]],
    results_action_dir: Path,
    *,
    top_k_retrieve: int = 10,
    query_strategy: Any = "template",
    rerank_mode: str = "crossencoder_colbert",
    run_rerank_ablation: bool = False,
    comparison_modes: Optional[List[str]] = None,
    index_embed_model: Optional[str] = None,
) -> None:
    """Run retrieval → optional rerank ablation → LLM. See module flags for stdout detail."""

    if _PIPELINE_COMPACT_STDOUT:
        print("=" * 80)
        print(
            f"Sample {sample.get('sample_id')} | "
            f"attack_type={_attack_type_for_display(sample)} | "
            f"confidence={float(sample.get('confidence', 0) or 0):.1%}"
        )
    else:
        _vprint("=" * 80)
        _vprint(
            f"Sample {sample.get('sample_id')} | "
            f"label={sample.get('predicted_label', 'UNKNOWN')} | "
            f"conf={sample.get('confidence', 0):.1%}"
        )

    q_template = build_template_rag_query(sample)
    q_rephrased = rephrase_template_query_deterministic(q_template)
    q_llm_concise, q_llm_expanded = build_llm_rag_queries(sample)

    generated_queries: Dict[str, str] = {
        "template": q_template,
        "rephrase": q_rephrased,
        "llm_concise": q_llm_concise,
        "llm_expanded": q_llm_expanded,
    }

    strategy_keys = _normalize_strategy_to_keys(query_strategy)
    if strategy_keys == ["multi"]:
        strategy_keys = ["template", "rephrase"]

    strategy_keys = [k for k in strategy_keys if k in generated_queries]
    if not strategy_keys:
        strategy_keys = ["template"]

    retrieval_queries = [generated_queries[k] for k in strategy_keys]
    retrieval_queries = [q for q in retrieval_queries if q] or [generated_queries["template"]]

    if not _PIPELINE_COMPACT_STDOUT:
        _vprint(f"Query strategy: {strategy_keys} | queries={len(retrieval_queries)}")

        _vprint("\nRetrieval queries used (vector search / MMR anchor):")
        for i, sk in enumerate(strategy_keys):
            q = generated_queries.get(sk, "")
            _vprint(f"  [{i + 1}] key={sk!r}")
            _vprint(f"      {q}")

        if _PIPELINE_SHOW_UNUSED_QUERY_VARIANTS:
            _vprint("\nOther query variants (not used this run):")
            for k in ("template", "rephrase", "llm_concise", "llm_expanded"):
                if k in strategy_keys:
                    continue
                _vprint(f"\n[{k}]\n{generated_queries.get(k, '')}")
        else:
            unused = [
                k
                for k in ("template", "rephrase", "llm_concise", "llm_expanded")
                if k not in strategy_keys
            ]
            if unused:
                _vprint(f"\n(Unused query variant keys this run: {', '.join(unused)})")

    if len(retrieval_queries) == 1:
        query_for_save = retrieval_queries[0]
    else:
        parts = [f"[{i+1}] {q}" for i, q in enumerate(retrieval_queries)]
        query_for_save = "MULTI_QUERY\n\n" + "\n\n---\n\n".join(parts)

    modes = (
        list(comparison_modes)
        if (run_rerank_ablation and comparison_modes)
        else None
    )
    if run_rerank_ablation:
        modes = modes or ["none", "bm25", "crossencoder", "colbert"]

    if run_rerank_ablation and modes:
        if not _PIPELINE_COMPACT_STDOUT:
            _vprint(
                f"\nRerank ablation: {len(modes)} mode(s) -> "
                f"{len(modes)} LLM call(s) for this sample."
            )
        modes_payload: Dict[str, Dict[str, Any]] = {}
        per_query_results: Optional[List[List[Dict[str, Any]]]] = None
        # Retrieve a wider reranked pool per mode (default 30) than the LLM
        # will ever see (top_k_retrieve, default 10). The LLM prompt uses
        # ``rag_results`` (top-K slice); the full pool is persisted under
        # ``rag_info.candidate_pool`` so Score.ipynb can evaluate Recall@k /
        # nDCG@k with a denominator larger than k (yielding fractional recall
        # across Dense-MMR / BM25 / cross-encoder / ColBERT).
        candidate_pool_size = max(int(top_k_retrieve), _RERANK_CANDIDATE_POOL_SIZE)

        for mode_i, mode in enumerate(modes):
            rag_candidate_pool, pq = retrieve_rag_context_multi(
                vector_store,
                retrieval_queries,
                top_k=candidate_pool_size,
                rerank_mode=str(mode),
            )
            rag_results = rag_candidate_pool[: int(top_k_retrieve)]
            if per_query_results is None:
                per_query_results = pq
                if (
                    not _PIPELINE_COMPACT_STDOUT
                    and _PIPELINE_PRINT_PER_QUERY_CHUNKS
                ):
                    _vprint("\nRAG retrieval results (per query, vector scores; before merge/MMR/rerank):")
                    for i, q in enumerate(retrieval_queries):
                        _vprint(f"\nQuery {i+1}/{len(retrieval_queries)}:\n{q}")
                        _print_doc_summary(
                            f"Retrieved {len(per_query_results[i])} docs (showing up to 5):",
                            per_query_results[i],
                        )

            label = _RERANK_MODE_LABELS.get(str(mode), str(mode))
            if _PIPELINE_COMPACT_STDOUT:
                _print_compact_top_sources_for_mode(
                    str(mode),
                    rag_results,
                    top_n=_PIPELINE_COMPACT_TOP_SOURCES,
                )
            else:
                _vprint("")
                _vprint("-" * 80)
                _vprint(f"RERANK MODE: {mode} — {label}")
                _vprint("-" * 80)
                if _PIPELINE_PRINT_MERGED_CHUNK_SUMMARY:
                    _print_doc_summary(
                        f"Merged sections (title + score + snippet; full text in RAG CONTEXT below): {len(rag_results)} total (showing up to 5):",
                        rag_results,
                    )
                _vprint("")

            prompt = create_prompt(
                sample, rag_results, attack_actions_data, agentic_features_data
            )
            if not _PIPELINE_COMPACT_STDOUT:
                _print_pipeline_prompt_artifacts(
                    rag_results,
                    prompt,
                    header_extra=f" [rerank_mode={mode}]",
                    ablation_mode_index=mode_i,
                    ablation_modes_total=len(modes),
                )
            if not _PIPELINE_COMPACT_STDOUT:
                print(f"Calling LLM API (rerank_mode={mode})...")
            llm_response = call_llm_api(prompt)
            if _PIPELINE_COMPACT_STDOUT:
                _print_llm_response_compact(llm_response)
            else:
                _print_llm_response_summary(llm_response)

            modes_payload[str(mode)] = {
                "label": label,
                "rag_results": rag_results,
                "candidate_pool": rag_candidate_pool,
                "llm_response": llm_response,
            }

        retrieval_entries = [
            {"strategy_key": sk, "query_text": generated_queries[sk]}
            for sk in strategy_keys
        ]
        try:
            cmp_path = save_rerank_comparison_report(
                sample,
                query_for_save,
                modes_payload,
                results_action_dir,
                retrieval_query_entries=retrieval_entries,
                index_embed_model=index_embed_model,
            )
            print(f"Saved rerank comparison report to {cmp_path.name}")
        except Exception as e:
            print(f"Error saving rerank comparison report: {e}")

        print("=" * 80)
        return

    rag_results, per_query_results = retrieve_rag_context_multi(
        vector_store,
        retrieval_queries,
        top_k=top_k_retrieve,
        rerank_mode=rerank_mode,
    )

    if not _PIPELINE_COMPACT_STDOUT and _PIPELINE_PRINT_PER_QUERY_CHUNKS:
        _vprint("\nRAG retrieval results:")
        for i, q in enumerate(retrieval_queries):
            _vprint(f"\nQuery {i+1}/{len(retrieval_queries)}:\n{q}")
            _print_doc_summary(
                f"Retrieved {len(per_query_results[i])} docs (showing up to 5):",
                per_query_results[i],
            )

    if _PIPELINE_COMPACT_STDOUT:
        _print_compact_top_sources_for_mode(
            str(rerank_mode),
            rag_results,
            top_n=_PIPELINE_COMPACT_TOP_SOURCES,
        )
    else:
        _vprint("")
        if _PIPELINE_PRINT_MERGED_CHUNK_SUMMARY:
            _print_doc_summary(
                f"Merged sections (title + score + snippet; full text in RAG CONTEXT below): {len(rag_results)} total (showing up to 5):",
                rag_results,
            )
        _vprint("")

    prompt = create_prompt(
        sample, rag_results, attack_actions_data, agentic_features_data
    )
    if not _PIPELINE_COMPACT_STDOUT:
        _print_pipeline_prompt_artifacts(rag_results, prompt)

    if not _PIPELINE_COMPACT_STDOUT:
        print("Calling LLM API...")
    llm_response = call_llm_api(prompt)

    if not llm_response:
        print("Failed to get LLM response")
        print("=" * 80)
        return

    if _PIPELINE_COMPACT_STDOUT:
        _print_llm_response_compact(llm_response)
    else:
        _print_llm_response_summary(llm_response)

    try:
        output_file = save_action_plan(
            sample,
            query_for_save,
            rag_results,
            llm_response,
            results_action_dir,
            variant=rerank_mode if rerank_mode != "crossencoder_colbert" else None,
        )
        print(f"Saved action plan to {output_file.name}")
    except Exception as e:
        print(f"Error saving action plan: {e}")

    print("=" * 80)

## Action pipeline driver

Run the cells below **in order** after the **retrieval / pipeline definitions** cell.

1. **Preload rerankers** — loads CrossEncoder and ColBERT once so loader noise stays out of the first sample trace.
2. **Configuration** — query strategy and rerank ablation flags (see also `_PIPELINE_*` toggles in the definitions cell for prompt/RAG verbosity).
3. **Execute pipeline** — retrieval, printed RAG context + prompts, LLM calls.


In [4]:
# Preload rerank models (optional but recommended before Execute).
ensure_rerankers_loaded()


C:\Users\nashe\AppData\Local\Temp\ipykernel_69588\2787919530.py:222: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel  # type: ignore


[Apr 21, 14:14:29] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
Rerankers loaded OK: CrossEncoder='cross-encoder/ms-marco-MiniLM-L-6-v2', ColBERT='colbert-ir/colbertv2.0'


C:\Projects\AML\VFL-RAG\.venv\Lib\site-packages\colbert\utils\amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
C:\Projects\AML\VFL-RAG\.venv\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


(CrossEncoder(
   (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'BertForSequenceClassification'})
 ),
 <ragatouille.RAGPretrainedModel.RAGPretrainedModel at 0x27642a171a0>)

In [5]:
# -----------------------------
# Retrieval / driver configuration
# -----------------------------
# Compact notebook output: attack type, top-N source|title per rerank mode, full LLM JSON.
_PIPELINE_COMPACT_STDOUT = True
_PIPELINE_COMPACT_TOP_SOURCES = 10
if _PIPELINE_COMPACT_STDOUT:
    _PIPELINE_SHOW_UNUSED_QUERY_VARIANTS = False
    _PIPELINE_PRINT_PER_QUERY_CHUNKS = False
    _PIPELINE_PRINT_MERGED_CHUNK_SUMMARY = False
    _PIPELINE_PRINT_RAG_CONTEXT_VERBATIM = False
    _PIPELINE_PRINT_FULL_LLM_PROMPT = False

QUERY_STRATEGY = "template"  # or: ["template", "rephrase"]

RUN_RERANK_ABLATION = True
RERANK_COMPARISON_MODES = ["none", "bm25", "crossencoder", "colbert"]
DEFAULT_RERANK_MODE = "crossencoder_colbert"

print("Driver configuration:")
print(f"  _PIPELINE_COMPACT_STDOUT = {_PIPELINE_COMPACT_STDOUT} (top {_PIPELINE_COMPACT_TOP_SOURCES} sources per rerank mode)")
print(f"  QUERY_STRATEGY = {QUERY_STRATEGY!r}")
print(f"  RUN_RERANK_ABLATION = {RUN_RERANK_ABLATION}")
print(f"  RERANK_COMPARISON_MODES = {RERANK_COMPARISON_MODES}")
print(f"  DEFAULT_RERANK_MODE = {DEFAULT_RERANK_MODE!r}")
print(f"  samples_for_actions = {len(samples_for_actions)} sample(s)")
print(f"  RESULTS_DIR = {RESULTS_DIR}")


Driver configuration:
  _PIPELINE_COMPACT_STDOUT = True (top 10 sources per rerank mode)
  QUERY_STRATEGY = 'template'
  RUN_RERANK_ABLATION = True
  RERANK_COMPARISON_MODES = ['none', 'bm25', 'crossencoder', 'colbert']
  DEFAULT_RERANK_MODE = 'crossencoder_colbert'
  samples_for_actions = 9 sample(s)
  RESULTS_DIR = RAG_docs\action_plans


In [6]:
# Execute: RAG + LLM for each queued sample.
for sample in samples_for_actions:
    run_agent_pipeline_for_sample(
        sample,
        vector_store,
        attack_actions_data,
        agentic_features_data,
        RESULTS_DIR,
        top_k_retrieve=10,
        query_strategy=QUERY_STRATEGY,
        rerank_mode=DEFAULT_RERANK_MODE,
        run_rerank_ablation=RUN_RERANK_ABLATION,
        comparison_modes=RERANK_COMPARISON_MODES,
        index_embed_model=_INDEX_EMBED_MODEL,
    )

print("Done. Action plans in:", RESULTS_DIR)


Sample 0 | attack_type=BENIGN | confidence=96.5%

Top 10 sources — none (no rerank (vector + MMR only))
  [1] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: Table 11 shows the assigned Scale of impact ratings for each component and interface . This rating is determined based on how a breach of a single component/interfaces would affect the entire network . Table 10: Service impact, Service impact and the Scale of
  [2] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: SMO, Non-RT RIC, rApps and associated interfaces have been assigned a “Low” rating . SMO is likely to be deployed on network operator premises or in large data centers that are physically secured .
  [3] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: The specifications would benefit significantly from  providing further details on how some of the recommendations have been derived . Where available, these recommen

C:\Projects\AML\VFL-RAG\.venv\Lib\site-packages\colbert\utils\amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
C:\Projects\AML\VFL-RAG\.venv\Lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.07it/s]



Top 10 sources — colbert (ColBERT rerank)
  [1] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: Table 11 shows the assigned Scale of impact ratings for each component and interface . This rating is determined based on how a breach of a single component/interfaces would affect the entire network . Table 10: Service impact, Service impact and the Scale of
  [2] CIS_Controls_Guide_v8.1.2_0325_v2.pdf | CIS_Controls_Guide_v8.1.2_0325_v2: CIS Controls v8.1.1 Control 18: Penetration Testing Guidelines . Perform periodic external penetration tests based on program requirements, no less than annually . Validate security measures after each penetration test .
  [3] NIST.SP.800-53r5.pdf | NIST.SP.800-53r5: Denial-of-service events may occur due to a variety of internal and external causes, such as an attack by an adversary or a lack of planning to support organizational needs . A variety of technologies are available to limit or eliminate the origination and 

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.07it/s]



Top 10 sources — colbert (ColBERT rerank)
  [1] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: The analysis shows that the largest percentage of  threats (27%) is related to denial of service, i.e., a compromise the availability, as illustrated in Figure 4 . Tampering (20%) is followed by Elevation of privilege (19%), Elevation of privilege (19%) and Sp
  [2] NIST.SP.800-53r5.pdf | NIST.SP.800-53r5: This publication is available free of charge from: https://doi.org/10.6028/NIST.800-53r5 . Perform a covert channel analysis to identify those aspects of communications within the  system that are potential avenues for covert channels . Estimate the maximum bandwidth of those chann
  [3] CIS_Controls_Guide_v8.1.2_0325_v2.pdf | CIS_Controls_Guide_v8.1.2_0325_v2: CIS Controls v8.1.1 Control 18: Penetration Testing Guidelines . Perform periodic external penetration tests based on program requirements, no less than annually . Validate security measures aft

...


Top 10 sources — colbert (ColBERT rerank)
  [1] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: Spec Spec Spec 7.2.1.1: Service Enumeration/ Network Boundary . 1: 20,000 ports, 20,500 ports, 10,000 devices, 100 ports, 100,000 users, 100 and 200 ports . 1-100: 100 ports; 1: 200 ports; 100: 100: 500 ports .
  [2] ATTACK_Design_and_Philosophy_March_2020.pdf | ATTACK_Design_and_Philosophy_March_2020: The MITRE Corporation, "Adversary Emulation Plans," MITRE ATT&CK, [Online]. Available: https://www.mitre.org/publications/technical-papers/finding-cyber-threats-with-attck-based-analytics .
  [3] NIST.SP.800-53r5.pdf | NIST.SP.800-53r5: Denial-of-service events may occur due to a variety of internal and external causes, such as an attack by an adversary or a lack of planning to support organizational needs . A variety of technologies are available to limit or eliminate the origination and effects of such events .
  [4] paper3.pdf | paper3: A.11.1.2*    .1,

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.04s/it]



Top 10 sources — colbert (ColBERT rerank)
  [1] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: SMO, Non-RT RIC, rApps and associated interfaces have been assigned a “Low” rating . SMO is likely to be deployed on network operator premises or in large data centers that are physically secured .
  [2] NIST.SP.800-53r5.pdf | NIST.SP.800-53r5: Denial-of-service events may occur due to a variety of internal and external causes, such as an attack by an adversary or a lack of planning to support organizational needs . A variety of technologies are available to limit or eliminate the origination and effects of such events .
  [3] CIS_Controls_Guide_v8.1.2_0325_v2.pdf | CIS_Controls_Guide_v8.1.2_0325_v2: CIS Controls v8.1.1 Control 18: Penetration Testing Guidelines . Perform periodic external penetration tests based on program requirements, no less than annually . Validate security measures after each penetration test .
  [4] 20220311114640pmwebology186-196

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.07it/s]



Top 10 sources — colbert (ColBERT rerank)
  [1] open_ran_security_report_full_report_0.pdf | open_ran_security_report_full_report_0: Table 11 shows the assigned Scale of impact ratings for each component and interface . This rating is determined based on how a breach of a single component/interfaces would affect the entire network . Table 10: Service impact, Service impact and the Scale of
  [2] NIST.SP.800-53r5.pdf | NIST.SP.800-53r5: This publication is available free of charge from: https://doi.org/10.6028/NIST.800-53r5 . Perform a covert channel analysis to identify those aspects of communications within the  system that are potential avenues for covert channels . Estimate the maximum bandwidth of those chann
  [3] ATTACK_Design_and_Philosophy_March_2020.pdf | ATTACK_Design_and_Philosophy_March_2020: The MITRE Corporation, "Adversary Emulation Plans," MITRE ATT&CK, [Online]. Available: https://www.mitre.org/publications/technical-papers/finding-cyber-threats-with-attck-based-analy

In [7]:
# RAG retrieval-quality scores (nine rerank comparison JSON files)
# Requires prior cells: RESULTS_DIR, attack_actions_data, agentic_features_data (may be None).
# Picks the newest rerank_comparison_sample_<id>_*.json per sample_id 0..8.
# Writes CSV with one row per sample. Per mode (none / crossencoder / colbert)
# we compute Recall@10 and nDCG@10 on the top-10 retrieved chunks using a
# weak graded-relevance signal against a deterministic reference string
# (0.5 * ROUGE-1 + 0.5 * SBERT cosine; relevant if >= 0.25). Same helpers as
# Score.ipynb so the two notebooks stay in sync.

from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

TOP_K = 10
_GRADE_REL_THRESHOLD = 0.25

_MODE_ORDER = ("none", "bm25", "crossencoder", "colbert")
_MODE_PREFIX = {
    "none": "none",
    "bm25": "bm25",
    "crossencoder": "crossencoder",
    "colbert": "colbert",
}

_ACTION_SYNONYMS = {
    "rate limiting": ["limit rate", "rate limit"],
    "traffic scrubbing": ["enable scrubbing", "scrub traffic"],
    "blackhole routing": ["blackhole route", "blackhole"],
    "ip blocking": ["block ip", "ip block"],
    "acl update": ["update acl", "acl change"],
    "syn cookies": ["enable syn cookies", "syn cookie"],
    "connection limiting": ["limit connections", "conn limit"],
    "account lockout": ["lock account", "lockout"],
    "mfa enforce": ["enforce mfa", "require mfa"],
    "credential throttle": ["throttle login", "login throttle"],
    "fail2ban block": ["fail2ban", "ban ip"],
    "waf rules": ["waf rule", "apply waf"],
    "virtual patching": ["virtual patch", "hot patch"],
    "isolate service": ["isolate", "quarantine service"],
    "captcha challenge": ["captcha", "challenge"],
    "reputation filter": ["reputation", "filter reputation"],
    "js challenge": ["js challenge", "javascript challenge"],
    "tarpitting": ["tarpit", "slow scan"],
    "port hardening": ["harden ports", "close ports"],
    "scan threshold": ["threshold scan", "scan threshold"],
    "auto scale": ["autoscale", "scale out"],
    "log only": ["log", "logging"],
    "monitor": ["monitoring", "observe"],
}


_sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
_rouge_scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)


def _norm_action(s: str) -> str:
    return " ".join((s or "").lower().strip().split())


def _build_action_to_agents(agentic: Optional[Dict[str, Any]]) -> Dict[str, List[str]]:
    mapping: Dict[str, List[str]] = defaultdict(list)
    if not isinstance(agentic, dict):
        return mapping
    agents = agentic.get("agents") if "agents" in agentic else agentic
    if not isinstance(agents, dict):
        return mapping
    for agent_name, info in agents.items():
        if not isinstance(info, dict):
            continue
        for cap in info.get("action_capabilities", []) or []:
            mapping[_norm_action(str(cap))].append(agent_name)
    return mapping


def build_reference_text_for_label(
    label: str,
    attack_options: Optional[Dict[str, Any]],
    agentic: Optional[Dict[str, Any]],
) -> str:
    """Deterministic gold reference string (same shape as Score.ipynb)."""
    ulabel = (label or "UNKNOWN").strip().upper()
    attacks = (attack_options or {}).get("attacks") or {}
    actions = attacks.get(ulabel) or attacks.get("OTHERS", []) or []
    actions_norm = [_norm_action(a) for a in actions]

    action_to_agents = _build_action_to_agents(agentic)

    agent_to_actions: Dict[str, List[str]] = defaultdict(list)
    unassigned: List[str] = []
    for a_raw, a in zip(actions, actions_norm):
        capable_agents = action_to_agents.get(a, [])
        if not capable_agents:
            unassigned.append(a_raw)
            continue
        expanded = [a_raw] + _ACTION_SYNONYMS.get(a, [])
        for ag in capable_agents:
            agent_to_actions[ag].extend(expanded)

    def uniq(seq: List[str]) -> List[str]:
        seen = set()
        out = []
        for x in seq:
            k = _norm_action(x)
            if k and k not in seen:
                seen.add(k)
                out.append(x)
        return out

    ran = ", ".join(uniq(agent_to_actions.get("RAN", []))) or "none"
    edge = ", ".join(uniq(agent_to_actions.get("Edge", []))) or "none"
    core = ", ".join(uniq(agent_to_actions.get("Core", []))) or "none"

    ref = (
        f"{ulabel} response: "
        f"RAN actions: {ran}. "
        f"Edge actions: {edge}. "
        f"Core actions: {core}."
    )
    if unassigned:
        ref += f" Unassigned actions: {', '.join(unassigned)}."
    return ref


def _rouge1_f(reference: str, candidate: str) -> float:
    if not reference or not candidate:
        return 0.0
    return float(_rouge_scorer.score(reference, candidate)["rouge1"].fmeasure)


def _sbert_cosine(reference: str, candidate: str) -> float:
    if not reference or not candidate:
        return 0.0
    embs = _sbert_model.encode([reference, candidate])
    return float(cosine_similarity([embs[0]], [embs[1]])[0][0])


def _graded_relevance(reference: str, chunk_text: str) -> float:
    if not chunk_text.strip():
        return 0.0
    rouge = _rouge1_f(reference, chunk_text)
    sem = _sbert_cosine(reference, chunk_text)
    return float(max(0.0, 0.5 * rouge + 0.5 * sem))


def _recall_at_k(reference: str, ranked_chunks: List[str], k: int = TOP_K) -> float:
    if not ranked_chunks:
        return 0.0
    grades = [_graded_relevance(reference, t) for t in ranked_chunks]
    total_relevant = sum(1 for g in grades if g >= _GRADE_REL_THRESHOLD)
    if total_relevant == 0:
        return 0.0
    retrieved_relevant = sum(1 for g in grades[:k] if g >= _GRADE_REL_THRESHOLD)
    return float(retrieved_relevant / total_relevant)


def _ndcg_at_k(reference: str, ranked_chunks: List[str], k: int = TOP_K) -> float:
    if not ranked_chunks:
        return 0.0
    grades = [_graded_relevance(reference, t) for t in ranked_chunks]
    k = min(k, len(grades))
    dcg = sum((2.0**rel - 1.0) / np.log2(i + 1.0) for i, rel in enumerate(grades[:k], start=1))
    ideal = sorted(grades, reverse=True)[:k]
    idcg = sum((2.0**rel - 1.0) / np.log2(i + 1.0) for i, rel in enumerate(ideal, start=1))
    return float(dcg / idcg) if idcg > 0 else 0.0


def _chunk_string(result: Dict[str, Any]) -> str:
    title = str(result.get("title", "") or "")
    text = str(result.get("text", "") or "")
    return f"{title}\n\n{text}".strip()


def _pick_latest_per_sample_id(paths: List[Path]) -> Dict[int, Path]:
    best: Dict[int, Tuple[float, Path]] = {}
    pat = re.compile(r"rerank_comparison_sample_(\d+)_")
    for p in paths:
        m = pat.search(p.name)
        if not m:
            continue
        sid = int(m.group(1))
        mt = p.stat().st_mtime
        prev = best.get(sid)
        if prev is None or mt > prev[0]:
            best[sid] = (mt, p)
    return {k: v[1] for k, v in best.items()}


def load_nine_rerank_reports(
    action_dir: Path,
    *,
    sample_ids: Optional[List[int]] = None,
) -> List[Tuple[int, Path]]:
    action_dir = Path(action_dir)
    paths = list(action_dir.glob("rerank_comparison_sample_*.json"))
    by_sid = _pick_latest_per_sample_id(paths)
    ids = sample_ids if sample_ids is not None else list(range(9))
    out: List[Tuple[int, Path]] = []
    for sid in ids:
        p = by_sid.get(sid)
        if p is None:
            raise FileNotFoundError(
                f"No rerank_comparison_sample_{sid}_*.json under {action_dir}"
            )
        out.append((sid, p))
    return sorted(out, key=lambda x: x[0])


def build_rag_scores_dataframe(
    report_paths: List[Tuple[int, Path]],
    attack_options: Optional[Dict[str, Any]],
    agentic: Optional[Dict[str, Any]],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for sid, path in report_paths:
        with open(path, encoding="utf-8") as f:
            rep = json.load(f)
        label = str((rep.get("prediction") or {}).get("predicted_label") or "UNKNOWN")
        ulabel = label.strip().upper()
        reference_text = build_reference_text_for_label(ulabel, attack_options, agentic)

        modes_data = rep.get("modes") or {}

        row: Dict[str, Any] = {
            "attack_group": f"sample_{sid}_{ulabel}",
            "sample_id": sid,
            "predicted_label": ulabel,
        }
        for mk in _MODE_ORDER:
            prefix = _MODE_PREFIX[mk]
            mode_block = modes_data.get(mk)
            if not isinstance(mode_block, dict):
                # Older ablation reports may not contain this mode yet (e.g. BM25 pre-regen).
                row[f"{prefix}_Recall@10"] = float("nan")
                row[f"{prefix}_nDCG@10"] = float("nan")
                continue
            top = ((mode_block.get("rag_info") or {}).get("top_results")) or []
            ranked_chunks = [_chunk_string(d) for d in top]
            recall = _recall_at_k(reference_text, ranked_chunks, k=TOP_K)
            ndcg = _ndcg_at_k(reference_text, ranked_chunks, k=TOP_K)
            row[f"{prefix}_Recall@10"] = round(recall, 4)
            row[f"{prefix}_nDCG@10"] = round(ndcg, 4)
        rows.append(row)

    col_order = ["attack_group", "sample_id", "predicted_label"]
    for mk in _MODE_ORDER:
        p = _MODE_PREFIX[mk]
        col_order.extend([f"{p}_Recall@10", f"{p}_nDCG@10"])
    df = pd.DataFrame(rows)
    return df[[c for c in col_order if c in df.columns]]


_action_dir = Path(RESULTS_DIR)
_reports = load_nine_rerank_reports(_action_dir)
_df_scores = build_rag_scores_dataframe(
    _reports,
    attack_actions_data,
    agentic_features_data if isinstance(agentic_features_data, dict) else None,
)
_csv_out = _action_dir / "rag_context_rerank_keyword_scores.csv"
_df_scores.to_csv(_csv_out, index=False)
print(f"Wrote {_csv_out} ({len(_df_scores)} rows).")
try:
    from IPython.display import display

    display(_df_scores)
except Exception:
    print(_df_scores.to_string())


Wrote RAG_docs\action_plans\rag_context_rerank_keyword_scores.csv (9 rows).


,attack_group,sample_id,predicted_label,none_Recall@10,none_nDCG@10,bm25_Recall@10,bm25_nDCG@10,crossencoder_Recall@10,crossencoder_nDCG@10,colbert_Recall@10,colbert_nDCG@10
0,sample_0_BENIGN,0,BENIGN,0.0,0.9870,0.0,0.9444,0.0,0.9563,0.0,0.9594
1,sample_1_BOT,1,BOT,0.0,0.9601,0.0,0.9196,0.0,0.9077,0.0,0.9038
2,sample_2_DDOS,2,DDOS,1.0,0.9861,0.0,0.9514,1.0,0.9714,1.0,0.9320
3,sample_3_DOS,3,DOS,1.0,0.8919,0.0,0.9728,1.0,0.8954,1.0,0.9010
4,sample_4_FTPPATATOR,4,FTPPATATOR,0.0,0.9058,0.0,0.9270,0.0,0.8657,0.0,0.9337
5,sample_5_OTHERS,5,OTHERS,1.0,0.9629,1.0,0.8958,1.0,0.9290,1.0,0.9113
6,sample_6_PORTSCAN,6,PORTSCAN,1.0,0.9775,1.0,0.9486,0.0,0.9114,1.0,0.9860
7,sample_7_SSHPATATOR,7,SSHPATATOR,0.0,0.8786,0.0,0.9257,0.0,0.8661,0.0,0.9042
8,sample_8_WEBATTACK,8,WEBATTACK,0.0,0.9609,0.0,0.9089,0.0,0.9121,0.0,0.9318


## Rerank report metrics (optional)

After generating `rerank_comparison_sample_*.json`, run the next cell (needs `vector_store` from the load cell).

The next cell reads reports from **`RERANK_METRICS_REPORT_DIR`** (default `RAG_docs/sample_for_analysis`). Set it to `RESULTS_DIR` if you want metrics on freshly written `action_plans/` exports instead.

New reports include `retrieval_queries`, `mmr_rerank_anchor_query`, `index_embed_model`, and per-chunk `parent_id` / `child_index` / `source_file` under `modes.*.rag_info.top_results`. They also include a wider `modes.*.rag_info.candidate_pool` (default 30 sections from the reranked pool) used by `Score.ipynb` to compute Recall@k / nDCG@k with a denominator larger than *k*.

In [8]:
# Optional: mean cosine(query, top-k chunks) + pairwise LLM field comparison per report.
try:
    from utils.rag_utils import embeddings_from_vector_store
except ImportError:
    # Older rag_utils or stale kernel: same logic as utils.rag_utils.embeddings_from_vector_store
    def embeddings_from_vector_store(vector_store):
        for attr in ("embedding_function", "embeddings"):
            e = getattr(vector_store, attr, None)
            if e is not None and hasattr(e, "embed_query") and hasattr(e, "embed_documents"):
                return e
        raise ValueError(
            "Vector store has no LangChain-compatible Embeddings instance "
            "(expected embedding_function or embeddings with embed_query/embed_documents)."
        )

from utils.rerank_metrics import downstream_comparison_for_report, embedding_metrics_for_report

# Directory containing rerank_comparison_sample_*.json (default: curated analysis copies).
RERANK_METRICS_REPORT_DIR = Path("RAG_docs/sample_for_analysis")
# RERANK_METRICS_REPORT_DIR = RESULTS_DIR  # use this for reports under action_plans/

_MET_TOPK = 10
_emb = embeddings_from_vector_store(vector_store)
_paths = sorted(RERANK_METRICS_REPORT_DIR.glob("rerank_comparison_sample_*.json"))
if not _paths:
    print("No rerank_comparison_sample_*.json in", RERANK_METRICS_REPORT_DIR.resolve())
else:
    _cos_acc: dict[str, list[float]] = {}
    for _p in _paths[:20]:
        import json as _json

        _rep = _json.loads(_p.read_text(encoding="utf-8"))
        if _rep.get("report_type") != "rerank_ablation_comparison":
            continue
        _e = embedding_metrics_for_report(_rep, _emb, top_k=_MET_TOPK)
        _d = downstream_comparison_for_report(_rep)
        print(f"\n{_p.name} | sample_id={_rep.get('sample_id')}")
        for _mk, _st in (_e.get("per_mode") or {}).items():
            v = float(_st.get("mean_top_k_cosine_query_chunk", 0.0))
            _cos_acc.setdefault(_mk, []).append(v)
            print(f"  cosine_mean_top{_MET_TOPK} [{_mk}]: {v:.4f}")
        for _row in (_d.get("pairwise") or [])[:3]:
            print("  pair:", _row)
    print("\n--- aggregate (up to 20 files) ---")
    for _mk, vs in sorted(_cos_acc.items()):
        print(f"  [{_mk}] mean {sum(vs)/len(vs):.4f} over {len(vs)} reports")


rerank_comparison_sample_0_20260421_131619_669992.json | sample_id=0
  cosine_mean_top10 [none]: 0.4808
  cosine_mean_top10 [bm25]: 0.4580
  cosine_mean_top10 [crossencoder]: 0.4289
  cosine_mean_top10 [colbert]: 0.4097
  pair: {'mode_a': 'none', 'mode_b': 'bm25', 'threat_level_match': True, 'execution_priority_match': True, 'knowledge_sources_match': True, 'primary_actions_jaccard': 1.0}
  pair: {'mode_a': 'none', 'mode_b': 'crossencoder', 'threat_level_match': True, 'execution_priority_match': True, 'knowledge_sources_match': True, 'primary_actions_jaccard': 1.0}
  pair: {'mode_a': 'none', 'mode_b': 'colbert', 'threat_level_match': True, 'execution_priority_match': True, 'knowledge_sources_match': True, 'primary_actions_jaccard': 1.0}

rerank_comparison_sample_1_20260421_131742_359340.json | sample_id=1
  cosine_mean_top10 [none]: 0.4179
  cosine_mean_top10 [bm25]: 0.3986
  cosine_mean_top10 [crossencoder]: 0.3720
  cosine_mean_top10 [colbert]: 0.3596
  pair: {'mode_a': 'none', 'mod

### LLM vs reference (Score.ipynb-style) — 9 × 3 rerank matrix

Uses `build_reference_text_for_label` + `overall_reasoning` per rerank mode on `RAG_docs/sample_for_analysis`. Rows = `sample_id` 0–8 with predicted label; columns = the three rerank methods.

**At the very end of the cell** (after the heatmaps):

- **TABLE 1** — For each rerank method, the **mean** of BERTScore F1, SBERT cosine, and ROUGE-1 **averaged over all 9 attacks** (same as taking column means of the 9×3 matrices).
- **TABLE 2** — For each of the **9 attack types**, which rerank method has the **highest BERTScore F1** on that row (tie → first column).

Needs prior **imports** (`Path`) and loads SBERT/BERTScore here (can take ~30–60s first run).

In [9]:
# Context utilization table — LLM overall_reasoning vs retrieved top-10 context,
# aggregated per rerank method. Rows: Dense MMR (none) / BM25 / Cross-encoder /
# ColBERT. Columns: Util_SBert (SBERT cosine) and Util_CRAG (3-valued truthfulness
# proxy). These are the same two metrics added to Score.ipynb as
# utilization_sbert_cosine / utilization_crag_proxy — they quantify how much of
# the retrieved evidence the LLM actually grounds its answer in.
# CRAG proxy mirrors Score.ipynb: +1 accurate, 0 missing, -1 incorrect
# (accurate if SBERT >= 0.55 or ROUGE-1 >= 0.12; otherwise -1 when an answer exists).

import pandas as pd

_MODE_DISPLAY = {
    "none": "Dense MMR",
    "bm25": "BM25",
    "crossencoder": "Cross-encoder",
    "colbert": "ColBERT",
}
_CRAG_SBERT_THRESHOLD = 0.55
_CRAG_ROUGE1_THRESHOLD = 0.12


def _extract_overall_reasoning(plan: Any) -> str:
    if not isinstance(plan, dict):
        return ""
    return str(plan.get("overall_reasoning", "") or "").strip()


def _crag_style_truthfulness(sbert: float, rouge1: float, has_answer: bool) -> float:
    if not has_answer:
        return 0.0
    if sbert >= _CRAG_SBERT_THRESHOLD or rouge1 >= _CRAG_ROUGE1_THRESHOLD:
        return 1.0
    return -1.0


_reports_final = load_nine_rerank_reports(Path(RESULTS_DIR))

_per_mode_sbert: Dict[str, List[float]] = {mk: [] for mk in _MODE_ORDER}
_per_mode_crag: Dict[str, List[float]] = {mk: [] for mk in _MODE_ORDER}

for _sid, _path in _reports_final:
    with open(_path, encoding="utf-8") as _f:
        _rep = json.load(_f)
    _modes = _rep.get("modes") or {}
    for _mk in _MODE_ORDER:
        _block = _modes.get(_mk) or {}
        _top = ((_block.get("rag_info") or {}).get("top_results")) or []
        _ctx = "\n\n".join(_chunk_string(d) for d in _top[:TOP_K]).strip()
        _reasoning = _extract_overall_reasoning(_block.get("llm_action_plan"))
        if not _ctx or not _reasoning:
            continue
        _sb = _sbert_cosine(_reasoning, _ctx)
        _r1 = _rouge1_f(_reasoning, _ctx)
        _crag = _crag_style_truthfulness(_sb, _r1, has_answer=True)
        _per_mode_sbert[_mk].append(_sb)
        _per_mode_crag[_mk].append(_crag)

_rows_out = []
for _mk in _MODE_ORDER:
    _sb_list = _per_mode_sbert[_mk]
    _crag_list = _per_mode_crag[_mk]
    _rows_out.append(
        {
            "Method": _MODE_DISPLAY[_mk],
            "Util_SBert": round(sum(_sb_list) / len(_sb_list), 4) if _sb_list else float("nan"),
            "Util_CRAG": round(sum(_crag_list) / len(_crag_list), 4) if _crag_list else float("nan"),
        }
    )

df_reasoning_vs_context = pd.DataFrame(_rows_out).set_index("Method")

print("Context Utilization — LLM overall_reasoning vs retrieved top-10 context (mean over 9 samples)")
print("Util_SBert = SBERT cosine(retrieved_context, overall_reasoning)")
print("Util_CRAG  = 3-valued truthfulness proxy (+1 accurate / 0 missing / -1 incorrect)")
print(df_reasoning_vs_context.to_string())

try:
    from IPython.display import display

    display(df_reasoning_vs_context)
except Exception:
    pass

LLM overall_reasoning vs retrieved top-10 context (mean over 9 samples)
               SBERT cosine    CRAG
Method                             
Dense MMR            0.4424 -0.7778
BM25                 0.4143 -1.0000
Cross-encoder        0.3739 -1.0000
ColBERT              0.4036 -1.0000


,SBERT cosine,CRAG
Method,,
Dense MMR,0.4424,-0.7778
BM25,0.4143,-1.0000
Cross-encoder,0.3739,-1.0000
ColBERT,0.4036,-1.0000
